In [ ]:
import pandas as pd
import dlt
import requests
import numpy as np
import os


In [ ]:
# 🔴 Q1: What version of dlt? 
print("dlt version:", dlt.__version__)

dlt version: 1.12.3


In [ ]:
# 🔴 Q2: Helper function to create a DataFrame from a list of dictionaries

@dlt.resource(write_disposition="replace", name="zoomcamp_data")
def zoomcamp_data():
    docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
    docs_response = requests.get(docs_url)
    documents_raw = docs_response.json()

    for course in documents_raw:
        course_name = course['course']

        for doc in course['documents']:
            doc['course'] = course_name
            yield doc

In [ ]:
# 🔴 Q2: Creat dlt pipeline with qdrant 

from dlt.destinations import qdrant
qdrant_destination = qdrant(
  qd_path="db.qdrant", 
)

pipeline = dlt.pipeline(
    pipeline_name="zoomcamp_pipeline",
    destination=qdrant_destination,
    dataset_name="zoomcamp_tagged_data"
)

load_info = pipeline.run(zoomcamp_data())
print(pipeline.last_trace)

2025-07-06 21:40:18,593|[WARNING]|83108|8578244352|dlt|pipeline.py|_state_to_props:1680|The destination dlt.destinations.duckdb:None in state differs from destination dlt.destinations.qdrant:qdrant in pipeline and will be ignored


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model_optimized.onnx:   0%|          | 0.00/133M [00:00<?, ?B/s]

Run started at 2025-07-07 04:40:18.597037+00:00 and COMPLETED in 13.74 seconds with 4 steps.
Step extract COMPLETED in 1.08 seconds.

Load package 1751863228.606302 is EXTRACTED and NOT YET LOADED to the destination and contains no failed jobs

Step normalize COMPLETED in 0.05 seconds.
Normalized data for the following tables:
- _dlt_pipeline_state: 1 row(s)
- zoomcamp_data: 948 row(s)

Load package 1751863228.606302 is NORMALIZED and NOT YET LOADED to the destination and contains no failed jobs

Step load COMPLETED in 2.59 seconds.
Pipeline zoomcamp_pipeline load step completed in 2.58 seconds
1 load package(s) were loaded to destination qdrant and into dataset zoomcamp_tagged_data
The qdrant destination used /Users/jenniferwang/own-LLM/db.qdrant location to store data
Load package 1751863228.606302 is LOADED and contains no failed jobs

Step run COMPLETED in 13.74 seconds.
Pipeline zoomcamp_pipeline load step completed in 2.58 seconds
1 load package(s) were loaded to destination qdra

In [8]:
# 🔴 Q3: Embedding model-- look at meta.json

# fast-bge-small-en